# Original organizer starter: unchanged verification

This notebook checks the public starter without your team's changes.
GPU: **T4 x2**. Internet: **On**. Choose **Run All**.
It downloads the fixed model and public dataset, trains base and starter solution,
then reports time, memory, eval loss and rule checks.

The original starter normally fails checks **3 and 4** because the example SwiGLU
kernels do not count as your own implementation. This is expected for this
verification notebook, not a submission-ready result.

This public repository needs no GitHub token. Main team development remains in
the repository created by the hackathon platform. Do not run simultaneous GPU jobs.


In [ ]:
REPO_URL = "https://github.com/MakazhanAlpamys/soup-kernel-challenge.git"
COMMIT = "c7c2fcd15972ce8b898d8d3ff732ee192c817ceb"


In [ ]:
import os, subprocess
gpus = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout
print(gpus)
assert "T4" in gpus, "Switch the accelerator to GPU T4 x2"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
!python -m pip install -q "soup-cli[train]==0.77.0" "transformers==5.16.1" "peft==0.20.0" "trl==0.29.1" "bitsandbytes==0.50.2" "accelerate==1.14.0" pytest

In [ ]:
import shutil, subprocess
from pathlib import Path
repo = Path('/kaggle/working/original_starter_notebook')
if repo.exists():
    raise RuntimeError('This verification directory already exists. Use a fresh Kaggle session to preserve existing results.')
subprocess.run(['git', 'clone', '-q', REPO_URL, str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', COMMIT], check=True)
print(subprocess.check_output(['git', '-C', str(repo), 'log', '-1', '--format=%H %s'], text=True))


In [ ]:
!cd /kaggle/working/original_starter_notebook && python task/prepare_data.py
!cd /kaggle/working/original_starter_notebook && python -m pytest solution -q

In [ ]:
!cd /kaggle/working/original_starter_notebook && python harness/run_task.py

In [ ]:
!cd /kaggle/working/original_starter_notebook && python harness/check_rules.py
!cd /kaggle/working/original_starter_notebook && python harness/score.py runs/result.json

While iterating, a quicker run of your solution alone (checks 1 and 5-8 need the base, so they fail
in this mode): `python harness/run_task.py --only solution --out runs_try`. Add `--profile` to get a
table of every CUDA kernel's GPU time in `runs/solution_profile/profile.json`. Logs of every run
are in `runs/<run>/train.log`.